# 00 - MIMII: probe the ZIP and extract one machine (Google Colab)

**Run the cells from top to bottom, one at a time.** Wait for each to finish (a green tick) before the next.

What this notebook does:
1. Checks the Colab machine (Python, disk, RAM).
2. Connects Google Drive (our permanent storage).
3. **Probe:** does Zenodo allow reading single files out of the big ZIP without downloading all of it?
4. **Plan A:** extract only Fan 00, as small mono 16-bit WAV files, and back them up to Drive.
5. **Plan B (only if the probe says so):** download the ZIP once to Colab's *temporary* disk, check it, extract, delete.

Nothing large is ever saved on your laptop.

## Step 1 - Check the Colab machine

In [ ]:
import os, platform, shutil, subprocess, sys
print("Python :", sys.version.split()[0])
print("System :", platform.platform())
total, used, free = shutil.disk_usage("/content")
print(f"Disk   : {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB (temporary: erased when the session ends)")
mem = {l.split(":")[0]: int(l.split()[1]) for l in open("/proc/meminfo") if ":" in l and len(l.split()) > 1}
print(f"RAM    : {mem['MemTotal'] / 1e6:.1f} GB total, {mem['MemAvailable'] / 1e6:.1f} GB available")
gpu_text = "none (fine for this notebook)"
if shutil.which("nvidia-smi"):
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    if gpu.returncode == 0:
        gpu_text = gpu.stdout.strip()
print("GPU    :", gpu_text)

## Step 2 - Install the two small libraries we need

In [ ]:
!pip -q install remotezip soundfile
import importlib.metadata as md
for name in ("remotezip", "soundfile", "numpy", "pandas"):
    print(f"{name:<10}", md.version(name))

## Step 3 - Connect Google Drive
A pop-up asks permission. Choose your Google account and click **Allow**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/smart-listener-data"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive folder ready:", DRIVE_ROOT)

## Step 4 - Upload our tool file
Click **Choose files** and pick `scripts/mimii_tool.py` from your project folder on your laptop.
(The cell skips this if the file is already here.)

In [ ]:
from google.colab import files
os.chdir("/content")
if not os.path.exists("/content/mimii_tool.py"):
    files.upload()
assert os.path.exists("/content/mimii_tool.py"), "mimii_tool.py was not uploaded - run this cell again"
print("Tool file is ready.")

## Step 5 - Settings
Change nothing now. Later you will change `LIMIT` to `0` for the full run.

In [ ]:
MACHINE_TYPE = "fan"
MACHINE_ID = "id_00"
ZIP_FILE = "6_dB_fan.zip"
ZIP_MD5 = "0890f7d3c2fd8448634e69ff1d66dd47"      # from the Zenodo page
URL = f"https://zenodo.org/records/3384388/files/{ZIP_FILE}?download=1"
WORK = "/content/work/mono"                        # temporary working folder on Colab
DRIVE_MONO = f"{DRIVE_ROOT}/interim/mono16k"       # permanent copy on Drive
LIMIT = 20          # 20 files per label = quick smoke test.  Set to 0 to extract ALL files.
print("Link:", URL)

## Step 6 - PROBE: can we read single files without downloading the whole ZIP?
Read the last line of the output:
* `RESULT: remote reading works. Use PLAN A` -> continue with Step 7.
* `PLAN B` -> skip down to the Plan B section at the bottom.

**Copy the whole output of this cell and send it to me.**

In [ ]:
!python /content/mimii_tool.py probe --url "{URL}"

## Step 7 - (Plan A) Restore earlier work from Drive, if any
The first time, this just says nothing to restore.

In [ ]:
import shutil
if os.path.isdir(DRIVE_MONO):
    shutil.copytree(DRIVE_MONO, WORK, dirs_exist_ok=True)
    print("Restored earlier work from Drive.")
else:
    print("Nothing to restore yet (first run).")

## Step 8 - (Plan A) Extract Fan 00 and check it
With `LIMIT = 20` this takes about a minute. With `LIMIT = 0` it processes every Fan 00 file and can take a long time;
it is safe to stop and run again, because finished files are skipped.

In [ ]:
!python /content/mimii_tool.py extract --url "{URL}" --machine-type {MACHINE_TYPE} --machine-id {MACHINE_ID} --out {WORK} --limit-per-label {LIMIT}
!python /content/mimii_tool.py verify --dir {WORK}

## Step 9 - Look at the inventory table

In [ ]:
import pandas as pd
inv = pd.read_csv(f"{WORK}/{MACHINE_TYPE}_{MACHINE_ID}_inventory.csv")
print("Files per label:"); print(inv.groupby("label").size().to_string())
print("\nOriginal recordings - sample rates:", sorted(inv["sr"].unique().tolist()), "| channels:", sorted(inv["channels"].unique().tolist()))
print("Duration (s)  min / max:", inv["duration_s"].min(), "/", inv["duration_s"].max())
bad = inv[inv["issues"].notna()]
print("Files with unusual properties:", len(bad))
if len(bad): print(bad[["source_member", "issues"]].head(10).to_string(index=False))
inv.head(3)

## Step 10 - Back up to Google Drive (and make a small sample file for your laptop)

In [ ]:
shutil.copytree(WORK, DRIVE_MONO, dirs_exist_ok=True)
print("Backed up to:", DRIVE_MONO)

n_wavs = sum(1 for _ in __import__("pathlib").Path(WORK).rglob("*.wav"))
if LIMIT > 0 and n_wavs <= 200:
    os.makedirs(f"{DRIVE_ROOT}/samples", exist_ok=True)
    archive = shutil.make_archive(f"/content/{MACHINE_TYPE}_{MACHINE_ID}_sample", "zip", root_dir=WORK)
    shutil.copy(archive, f"{DRIVE_ROOT}/samples/")
    print("Small sample for your laptop:", f"{DRIVE_ROOT}/samples/{os.path.basename(archive)}",
          f"({os.path.getsize(archive) / 1e6:.1f} MB)")
else:
    print("No sample file made (only done for the small smoke test).")

---
# FULL RUN (Plan A) - all 8 machines: fan and pump, ids 00, 02, 04, 06
Only do this after the small test above worked. It is safe to stop and run again: finished files are skipped.
Everything is backed up to Drive after each machine, so a Colab disconnect loses nothing.

In [ ]:
import subprocess
ZIPS = {"fan": ("6_dB_fan.zip", "0890f7d3c2fd8448634e69ff1d66dd47"),
        "pump": ("6_dB_pump.zip", "a09ba6060c10fc09cd4c8770213b0b9f")}
IDS = ["id_00", "id_02", "id_04", "id_06"]
BASE = "https://zenodo.org/records/3384388/files"
def run(cmd):
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)

for mtype, (zname, _) in ZIPS.items():
    for mid in IDS:
        run(["python", "/content/mimii_tool.py", "extract", "--url", f"{BASE}/{zname}?download=1",
             "--machine-type", mtype, "--machine-id", mid, "--out", WORK, "--limit-per-label", "0"])
        shutil.copytree(WORK, DRIVE_MONO, dirs_exist_ok=True)
        print(f"[backed up to Drive] {mtype} {mid}")
r = subprocess.run(["python", "/content/mimii_tool.py", "verify", "--dir", WORK])
if r.returncode:
    print("\nNOTE: verify listed unusual files. That is OK here: the audit step on your laptop decides what to exclude.")

---
# PLAN B - only if Step 6 said the ZIP cannot be read remotely
This downloads the whole ZIP **once** to Colab's temporary disk (about 11 GB), checks it is not corrupted,
extracts only Fan 00, and then you can delete it. It never touches Google Drive or your laptop.

In [ ]:
ZIPS_DIR = "/content/zips"
os.makedirs(ZIPS_DIR, exist_ok=True)
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"Free Colab disk: {free_gb:.1f} GB (the ZIP needs about 11 GB)")
assert free_gb > 14, "Not enough temporary disk space - send me this number"
!wget -c -q --show-progress -O {ZIPS_DIR}/{ZIP_FILE} "{URL}"

In [ ]:
out = !md5sum {ZIPS_DIR}/{ZIP_FILE}
got = out[0].split()[0]
print("Got     :", got)
print("Expected:", ZIP_MD5)
print("CHECKSUM", "MATCHES - the ZIP is intact" if got == ZIP_MD5 else "DOES NOT MATCH - delete the file and download again")

In [ ]:
!python /content/mimii_tool.py probe --local-zip {ZIPS_DIR}/{ZIP_FILE}
!python /content/mimii_tool.py extract --local-zip {ZIPS_DIR}/{ZIP_FILE} --machine-type {MACHINE_TYPE} --machine-id {MACHINE_ID} --out {WORK} --limit-per-label {LIMIT}
!python /content/mimii_tool.py verify --dir {WORK}

In [ ]:
# Run this when you no longer need the ZIP (frees about 11 GB of temporary space)
os.remove(f"{ZIPS_DIR}/{ZIP_FILE}")
print("ZIP deleted.")

## PLAN B, full run - all 8 machines
Downloads one ZIP at a time, checks its md5, extracts all four machines from it, backs up to Drive, then deletes the ZIP.

In [ ]:
import hashlib
for mtype, (zname, want) in ZIPS.items():
    zpath = f"{ZIPS_DIR}/{zname}"
    assert shutil.disk_usage("/content").free / 1e9 > 14, "Not enough temporary disk space"
    run(["wget", "-c", "-q", "-O", zpath, f"{BASE}/{zname}?download=1"])
    h = hashlib.md5()
    with open(zpath, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""):
            h.update(chunk)
    assert h.hexdigest() == want, f"{zname}: md5 {h.hexdigest()} does not match {want}. Delete it and download again."
    print(f"{zname}: md5 OK")
    for mid in IDS:
        run(["python", "/content/mimii_tool.py", "extract", "--local-zip", zpath, "--machine-type", mtype,
             "--machine-id", mid, "--out", WORK, "--limit-per-label", "0"])
    shutil.copytree(WORK, DRIVE_MONO, dirs_exist_ok=True)
    os.remove(zpath)
    print(f"[done] {mtype}: backed up to Drive, ZIP deleted")
r = subprocess.run(["python", "/content/mimii_tool.py", "verify", "--dir", WORK])
if r.returncode:
    print("\nNOTE: verify listed unusual files. That is OK here: the audit step on your laptop decides what to exclude.")